<a href="https://colab.research.google.com/github/AqsaBatool256/flyrank-ml-internship-Aqsa-Batool-Saqib/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AqsaBatool256/flyrank-ml-internship-Aqsa-Batool-Saqib/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

## Rule
I will prioritize pages for refresh review when they are both stale and visibly active.
A page is stale when it has not been updated for 180 days or more.
A page is visible when it has at least 500 impressions in the last 90 days.
Reason code: stale_visible_page
Action: REFRESH

In [5]:
# TWO SIGNAL CHECKS

# Signal 1: freshness
df["freshness_bucket"] = pd.cut(
    df["days_since_last_update"],
    bins=[-np.inf, 89, 179, 364, np.inf],
    labels=["<90", "90-179", "180-364", "365+"]
)

freshness_table = (
    df.groupby("freshness_bucket", observed=False)
      .agg(
          n=("content_id", "size"),
          median_impressions=("impressions_90d", "median")
      )
      .reset_index()
)

print("SIGNAL 1 — FRESHNESS")
display(freshness_table)


# Signal 2: volume / visibility
df["volume_bucket"] = pd.cut(
    df["impressions_90d"],
    bins=[-np.inf, 499, 2999, 9999, np.inf],
    labels=["<500", "500-2999", "3000-9999", "10000+"]
)

volume_table = (
    df.groupby("volume_bucket", observed=False)
      .agg(
          n=("content_id", "size"),
          median_days_since_update=("days_since_last_update", "median")
      )
      .reset_index()
)

print("SIGNAL 2 — VISIBILITY / VOLUME")
display(volume_table)


# Simple verdicts
old_pages = df.loc[
    df["days_since_last_update"] >= 180,
    "impressions_90d"
].median()

recent_pages = df.loc[
    df["days_since_last_update"] < 180,
    "impressions_90d"
].median()

high_volume = df.loc[
    df["impressions_90d"] >= 500,
    "days_since_last_update"
].median()

low_volume = df.loc[
    df["impressions_90d"] < 500,
    "days_since_last_update"
].median()

freshness_verdict = (
    "CONFIRMED" if old_pages > recent_pages
    else "OPPOSITE" if old_pages < recent_pages
    else "MIXED"
)

volume_verdict = (
    "CONFIRMED" if high_volume >= low_volume
    else "OPPOSITE"
)

print("Freshness verdict:", freshness_verdict)
print("Visibility verdict:", volume_verdict)

SIGNAL 1 — FRESHNESS


,freshness_bucket,n,median_impressions
0,<90,20655,472.0
1,90-179,9171,1692.0
2,180-364,169,16.0
3,365+,5,2.0


SIGNAL 2 — VISIBILITY / VOLUME


,volume_bucket,n,median_days_since_update
0,<500,13274,20.0
1,500-2999,8443,22.0
2,3000-9999,4681,22.0
3,10000+,3602,25.0


Freshness verdict: OPPOSITE
Visibility verdict: CONFIRMED


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [6]:
import pandas as pd
import numpy as np
import os

DATA_URL = "https://raw.githubusercontent.com/AqsaBatool256/flyrank-ml-internship-Aqsa-Batool-Saqib/main/data/raw/content_refresh_anonymized.csv"

df = pd.read_csv(DATA_URL)

print("Rows:", len(df))
print("Columns:", len(df.columns))

df = df.drop_duplicates("content_id").copy()
df = df[df["impressions_90d"] > 0].copy()

print("Usable rows:", len(df))

# Signal 1: freshness
df["stale"] = (df["days_since_last_update"] >= 180).astype(int)

# Signal 2: visibility
df["visible"] = (df["impressions_90d"] >= 500).astype(int)

# One transparent baseline score
df["score"] = (
    df["stale"]
    * df["visible"]
    * np.log1p(df["impressions_90d"])
)

# One reason code
df["reason_code"] = np.where(
    (df["stale"] == 1) & (df["visible"] == 1),
    "stale_visible_page",
    "monitor"
)

# Action label
df["action"] = np.where(
    df["reason_code"] == "stale_visible_page",
    "REFRESH",
    "MONITOR"
)

# Rank
df = df.sort_values(
    ["score", "impressions_90d"],
    ascending=[False, False]
).reset_index(drop=True)

df["rank"] = np.arange(1, len(df) + 1)

# Create output folder
os.makedirs("/content/work/outputs", exist_ok=True)

# Write required CSV
output_path = "/content/work/outputs/baseline_action_score.csv"

queue = df[
    [
        "rank",
        "content_id",
        "score",
        "reason_code",
        "action",
        "days_since_last_update",
        "impressions_90d",
        "ctr",
        "avg_position"
    ]
].copy()

queue.to_csv(output_path, index=False)

print("CSV written successfully:")
print(output_path)

print("\nTOP 10:")
display(queue.head(10))


Rows: 30000
Columns: 44
Usable rows: 30000
CSV written successfully:
/content/work/outputs/baseline_action_score.csv

TOP 10:


,rank,content_id,score,reason_code,action,days_since_last_update,impressions_90d,ctr,avg_position
0,1,content_cf56e2e2e282,11.029699,stale_visible_page,REFRESH,194,61678,0.15,19.7
1,2,content_7368877ea310,10.993278,stale_visible_page,REFRESH,194,59472,0.13,24.8
2,3,content_1bfaa38ff26c,10.154869,stale_visible_page,REFRESH,194,25715,0.23,22.2
3,4,content_0a91db491d14,9.495519,stale_visible_page,REFRESH,193,13299,0.49,10.5
4,5,content_5feee3994adb,8.963544,stale_visible_page,REFRESH,194,7812,0.01,39.0
5,6,content_c2d929d83eaa,8.930494,stale_visible_page,REFRESH,193,7558,0.20,17.9
6,7,content_b16bd7307b39,8.431853,stale_visible_page,REFRESH,194,4590,0.00,31.0
7,8,content_fe16a55cd13d,8.424420,stale_visible_page,REFRESH,194,4556,0.33,16.4
8,9,content_ecb6215e79fd,8.396155,stale_visible_page,REFRESH,194,4429,0.38,25.3
9,10,content_928af3e22c80,7.437206,stale_visible_page,REFRESH,193,1697,0.12,15.8


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

 reviewed the ten highest-ranked pages from the baseline queue.

For each page, I record the action, why it was selected, and what could make the recommendation wrong.

In [7]:
# TOP-10 SKEPTICAL REVIEW

top10 = queue.head(10).copy()

review_rows = []

for _, row in top10.iterrows():

    if row["action"] == "REFRESH":
        why = (
            f"Stale for {int(row['days_since_last_update'])} days "
            f"with {int(row['impressions_90d'])} impressions."
        )

        wrong = (
            "Traffic may be seasonal, the page may no longer be strategically important, "
            "or freshness may not be the real cause of the opportunity."
        )

    else:
        why = "The page does not meet the baseline refresh thresholds."

        wrong = (
            "The page may still have an opportunity that this simple rule does not capture."
        )

    review_rows.append({
        "rank": int(row["rank"]),
        "action": row["action"],
        "reason_code": row["reason_code"],
        "why_it_is_here": why,
        "what_would_make_it_wrong": wrong
    })

review = pd.DataFrame(review_rows)

display(review)

,rank,action,reason_code,why_it_is_here,what_would_make_it_wrong
0,1,REFRESH,stale_visible_page,Stale for 194 days with 61678 impressions.,"Traffic may be seasonal, the page may no longe..."
1,2,REFRESH,stale_visible_page,Stale for 194 days with 59472 impressions.,"Traffic may be seasonal, the page may no longe..."
2,3,REFRESH,stale_visible_page,Stale for 194 days with 25715 impressions.,"Traffic may be seasonal, the page may no longe..."
3,4,REFRESH,stale_visible_page,Stale for 193 days with 13299 impressions.,"Traffic may be seasonal, the page may no longe..."
4,5,REFRESH,stale_visible_page,Stale for 194 days with 7812 impressions.,"Traffic may be seasonal, the page may no longe..."
5,6,REFRESH,stale_visible_page,Stale for 193 days with 7558 impressions.,"Traffic may be seasonal, the page may no longe..."
6,7,REFRESH,stale_visible_page,Stale for 194 days with 4590 impressions.,"Traffic may be seasonal, the page may no longe..."
7,8,REFRESH,stale_visible_page,Stale for 194 days with 4556 impressions.,"Traffic may be seasonal, the page may no longe..."
8,9,REFRESH,stale_visible_page,Stale for 194 days with 4429 impressions.,"Traffic may be seasonal, the page may no longe..."
9,10,REFRESH,stale_visible_page,Stale for 193 days with 1697 impressions.,"Traffic may be seasonal, the page may no longe..."


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.